# Fake Job Detection in Pakistan
## Model Training Notebook

Libraries used: **Pandas, NumPy, Seaborn, Scikit-Learn**.

Steps: load data, handle missing values, clean & combine text, encode the label,
convert text to numbers (TF-IDF), scale, split, train 3 models
(Logistic Regression, KNN, SVM), cross-validate, compare, and save the best model.

### 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import joblib, json

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# The 3 models we must use
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix)

sns.set(style='whitegrid')


### 2. Settings  (CHANGE THESE TO MATCH YOUR DATASET)
- `DATASET_PATH`: where your CSV is.
- `TARGET_COL`: the column that says fake or real.
- `TEXT_COLS`: the text columns to use (only the ones that exist will be used).

In [ ]:
DATASET_PATH = '../data/fake_jobs_dataset.csv'
TARGET_COL   = 'fraudulent'   # 1 = Fake, 0 = Real (change if your dataset differs)
TEXT_COLS    = ['title', 'company_profile', 'description', 'requirements', 'benefits']


### 3. Load the dataset

In [ ]:
df = pd.read_csv(DATASET_PATH)
print('Rows and columns:', df.shape)
df.head()


### 4. Look at the data and missing values

In [ ]:
print(df.info())
print()
print('Missing values per column:')
print(df.isnull().sum())


Visualise the missing values with a Seaborn heatmap.

In [ ]:
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missing Values (yellow = missing)')
plt.show()


### 5. Target distribution (Real vs Fake)
Fake jobs are usually a small minority, so we expect an imbalance.

In [ ]:
# Remove rows where the label is missing
df = df.dropna(subset=[TARGET_COL])

print(df[TARGET_COL].value_counts())

plt.figure(figsize=(5, 4))
sns.countplot(x=TARGET_COL, data=df, palette='Set2')
plt.title('How many Real vs Fake postings')
plt.show()


### 6. Handle missing values in the text columns
We only keep the text columns that actually exist, and fill blanks with empty text.

In [ ]:
text_cols = [c for c in TEXT_COLS if c in df.columns]
print('Using these text columns:', text_cols)

for c in text_cols:
    df[c] = df[c].fillna('')


### 7. Combine and clean the text
We join the text columns into one column, then clean it.

In [ ]:
import re

def clean_text(text):
    # Clean text the SAME way the backend does.
    text = str(text).lower()
    text = re.sub(r"<[^>]*>", " ", text)     # remove html tags
    text = re.sub(r"[^a-z ]", " ", text)      # keep letters and spaces only
    text = re.sub(r" +", " ", text).strip()   # collapse multiple spaces
    return text


In [ ]:
# Join all text columns into one string per row
df['combined_text'] = df[text_cols].agg(' '.join, axis=1)

# Clean the combined text
df['clean_text'] = df['combined_text'].apply(clean_text)

df[['combined_text', 'clean_text']].head(3)


### 8. Encode the target into readable labels
We turn the label into the words **Real** and **Fake** so results are easy to read.

In [ ]:
values = set(df[TARGET_COL].dropna().unique())
if values <= {0, 1}:
    y = df[TARGET_COL].map({1: 'Fake', 0: 'Real'})
else:
    # If your labels are already text, they are used as-is.
    y = df[TARGET_COL].astype(str)

X = df['clean_text']
print(y.value_counts())


### 9. Train-Test Split
80% for training, 20% for testing. `stratify=y` keeps the same Real/Fake ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print('Train size:', len(X_train))
print('Test size :', len(X_test))


### 10. Define the 3 models
Each model is wrapped in a Pipeline: **TF-IDF (text -> numbers) -> Scaling -> Model**.
TF-IDF is how we *encode* the text. The scaler keeps features on a similar range
(helpful for KNN and SVM). `class_weight='balanced'` helps with the Fake/Real imbalance.

In [ ]:
def make_pipeline(model):
    return Pipeline([
        ('tfidf', TfidfVectorizer(stop_words='english', max_features=5000, ngram_range=(1, 2))),
        ('scaler', StandardScaler(with_mean=False)),
        ('model', model)
    ])

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'SVM': SVC(kernel='linear', probability=True, class_weight='balanced'),
}
print('Models ready:', list(models.keys()))


### 11. Train, cross-validate and evaluate each model
For fraud detection, **Recall and F1 on the Fake class** matter most.
*(SVM may take a minute or two.)*

In [ ]:
results = []
trained = {}
conf_matrices = {}

for name, model in models.items():
    pipe = make_pipeline(model)

    # 5-fold cross validation on the training data
    cv_score = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

    # Train on the full training set
    pipe.fit(X_train, y_train)

    train_acc = accuracy_score(y_train, pipe.predict(X_train))
    y_pred = pipe.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, pos_label='Fake', zero_division=0)
    rec  = recall_score(y_test, y_pred, pos_label='Fake', zero_division=0)
    f1   = f1_score(y_test, y_pred, pos_label='Fake', zero_division=0)

    results.append({
        'model': name,
        'train_accuracy': round(train_acc, 4),
        'test_accuracy': round(test_acc, 4),
        'cv_accuracy': round(cv_score, 4),
        'precision': round(prec, 4),
        'recall': round(rec, 4),
        'f1': round(f1, 4),
    })
    trained[name] = pipe
    conf_matrices[name] = confusion_matrix(y_test, y_pred, labels=['Real', 'Fake'])
    print(f'{name:20s} | Train: {train_acc:.3f} | Test: {test_acc:.3f} | CV: {cv_score:.3f} | F1(Fake): {f1:.3f}')


### 12. Model Comparison Table

In [ ]:
results_df = pd.DataFrame(results).sort_values('cv_accuracy', ascending=False).reset_index(drop=True)
results_df


### 13. Accuracy Comparison Chart

In [ ]:
plot_data = results_df.melt(
    id_vars='model',
    value_vars=['train_accuracy', 'test_accuracy', 'cv_accuracy'],
    var_name='Type', value_name='Accuracy'
)
plt.figure(figsize=(8, 5))
sns.barplot(data=plot_data, x='model', y='Accuracy', hue='Type', palette='Set2')
plt.title('Accuracy Comparison of Models')
plt.ylim(0, 1)
plt.legend(title='')
plt.show()


### 14. Confusion Matrix for each model

In [ ]:
for name in models:
    plt.figure(figsize=(4, 3))
    sns.heatmap(conf_matrices[name], annot=True, fmt='d', cmap='Greens',
                xticklabels=['Real', 'Fake'], yticklabels=['Real', 'Fake'])
    plt.title('Confusion Matrix - ' + name)
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()


### 15. Automatically select and save the best model
We pick the model with the highest cross-validation accuracy and save it as
`best_model.pkl`. We also save the comparison to `model_results.json`.

In [ ]:
best_name = results_df.iloc[0]['model']
best_model = trained[best_name]
best_row = results_df.iloc[0]

print('Best model:', best_name)
print('Training Accuracy   :', round(best_row['train_accuracy'] * 100, 2), '%')
print('Testing Accuracy    :', round(best_row['test_accuracy'] * 100, 2), '%')
print('Cross Validation    :', round(best_row['cv_accuracy'] * 100, 2), '%')

# Save the model so the Flask app can use it
joblib.dump(best_model, '../model/best_model.pkl')
print('Saved ../model/best_model.pkl')

# Save the comparison numbers (as percentages) for the website / report
descriptions = {
    'Logistic Regression': 'A simple linear model that estimates the probability a job is fake.',
    'KNN': 'Classifies a posting by looking at the most similar past postings.',
    'SVM': 'Finds the best boundary that separates real and fake postings.',
}
models_out = []
for r in results:
    models_out.append({
        'name': r['model'],
        'description': descriptions.get(r['model'], ''),
        'train_accuracy': round(r['train_accuracy'] * 100, 2),
        'test_accuracy': round(r['test_accuracy'] * 100, 2),
        'cv_accuracy': round(r['cv_accuracy'] * 100, 2),
        'precision': round(r['precision'] * 100, 2),
        'recall': round(r['recall'] * 100, 2),
        'f1': round(r['f1'] * 100, 2),
    })
models_out.sort(key=lambda m: m['cv_accuracy'], reverse=True)
with open('../model/model_results.json', 'w') as f:
    json.dump({'best_model': best_name, 'test_size': int(len(y_test)), 'models': models_out}, f, indent=2)
print('Saved ../model/model_results.json')


Training finished. Open **model_evaluation.ipynb** for a detailed evaluation
of the saved best model.